In [9]:
import os
import numpy as np
from PIL import Image

class PatternRecognitionSystem:
    def __init__(self, grid_size=(5, 5)):
        self.grid_size = grid_size
        self.classes_data = {}  
        self.cluster_centers = {} 

    def extract_features(self, image_path):
        img = Image.open(image_path).convert('L')
        img_array = np.array(img)
        binary_img = (img_array < 128).astype(int)
        
        h, w = binary_img.shape
        rows, cols = self.grid_size
        
        cell_h = h // rows
        cell_w = w // cols
        
        absolute_vector = []
        for i in range(rows):
            for j in range(cols):
                cell = binary_img[i*cell_h:(i+1)*cell_h, j*cell_w:(j+1)*cell_w]
                absolute_vector.append(np.sum(cell))
                
        absolute_vector = np.array(absolute_vector)
        max_pixels_in_cell = cell_h * cell_w
        normalized_vector = absolute_vector / max_pixels_in_cell
        return absolute_vector, normalized_vector

    def train_system(self, training_data):
        for class_name, image_paths in training_data.items():
            vectors = []
            for path in image_paths:
                _, norm_vec = self.extract_features(path)
                vectors.append(norm_vec)
            
            self.classes_data[class_name] = vectors
            self.cluster_centers[class_name] = np.mean(vectors, axis=0)

    def calculate_distance(self, v1, v2):
       return np.sqrt(np.sum((v1-v2)**2))

    def classify(self, image_path, metric='euclidean'):
        _, norm_vector = self.extract_features(image_path)
        best_match = None
        min_distance = float('inf')
        distances = {}
        for class_name, center in self.cluster_centers.items():
            dist = self.calculate_distance(norm_vector, center)
            distances[class_name] = dist
            if dist < min_distance:
                min_distance = dist
                best_match = class_name
        return best_match, distances, norm_vector


if __name__ == "__main__":
    system = PatternRecognitionSystem(grid_size=(5, 5))
    
    try:
        training_set = {
            "1": [f"bmp-shki/1/{f}" for f in os.listdir("bmp-shki/1") if f.endswith(".bmp")], 
            "2": [f"bmp-shki/2/{f}" for f in os.listdir("bmp-shki/2") if f.endswith(".bmp")],
            "3": [f"bmp-shki/3/{f}" for f in os.listdir("bmp-shki/3") if f.endswith(".bmp")],
            "4": [f"bmp-shki/4/{f}" for f in os.listdir("bmp-shki/4") if f.endswith(".bmp")],
            "5": [f"bmp-shki/5/{f}" for f in os.listdir("bmp-shki/5") if f.endswith(".bmp")]
        }
        
        for cls, files in training_set.items():
            if not files:
                print(f"Попередження: У папці для {cls} не знайдено жодного файлу .bmp")

        print("Навчання системи")
        system.train_system(training_set)
        print("Навчання успішне!\n")
        
        test_image = "test.bmp" 
        
        print(f"Класифікація файлу: {test_image}")
        result_class, dists, vec = system.classify(test_image, metric='euclidean')
        
        print(f"\nРезультат: образ схожий на '{result_class}'")
        
            
    except FileNotFoundError as e:
        print(f"\nПОМИЛКА: {e}")
        print("Переконайтеся, що ви створили файл 'test.bmp")

Навчання системи
Навчання успішне!

Класифікація файлу: test.bmp

Результат: образ схожий на '5'
